# Exportar YOLO `.pt` a TensorRT `.engine`

Este notebook tiene un objetivo: exportar un checkpoint YOLO entrenado (`.pt`) a TensorRT (`.engine`) y validarlo rápidamente.


## 1. Qué Hace Este Notebook

- Valida disponibilidad de GPU/CUDA/TensorRT.
- Exporta un modelo YOLO `.pt` a TensorRT `.engine`.
- Ejecuta una predicción rápida opcional.
- Incluye una comparación rápida opcional de velocidad (`.pt` vs `.engine`).


## 2. Revisión del Entorno

Ejecuta esto primero. Si falta TensorRT, instálalo en el entorno antes de exportar.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import torch

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch version: {torch.__version__}")


In [ ]:
# Opcional: revisión rápida de driver/GPU
!nvidia-smi


In [ ]:
# Revisión de versión de TensorRT; requerida para exportar/inferir con .engine
import tensorrt
print(f"TensorRT version: {tensorrt.__version__}")


## 3. Configurar Rutas y Parámetros de Exportación

Edita solo esta celda para cada exportación.


In [ ]:

from ultralytics import YOLO

DATA_ROOT = PROJECT_ROOT / "data"


def slug(*parts: str | None) -> str:
    tokens = []
    for part in parts:
        if part is None:
            continue
        text = str(part).strip().lower().replace("-", "_").replace(" ", "_")
        text = "_".join(chunk for chunk in text.split("_") if chunk)
        if text:
            tokens.append(text)
    return "_".join(tokens)


def dataset_base_dir(mine: str, mine_code: str | None = None) -> Path:
    mine_key = mine.strip().upper()
    if mine_key == "BONANZA":
        return DATA_ROOT / "BONANZA"
    if mine_key == "TENGEL":
        if not mine_code:
            raise ValueError("TENGEL requiere mine_code, por ejemplo 'M17' o 'M93'.")
        return DATA_ROOT / "TENGEL" / mine_code.upper()
    raise ValueError(f"Mina no soportada: {mine}")


def model_output_dir(
    mine: str,
    mine_code: str | None,
    model_tag: str,
    version: str,
    run_date: str,
    aug_tag: str = "no_aug",
    model_group: str | None = None,
    model_dir_name: str | None = None,
) -> Path:
    base = dataset_base_dir(mine, mine_code) / "models"
    mine_key = mine.strip().upper()
    if mine_key == "BONANZA" and model_group:
        base = base / slug(model_group)
    if model_dir_name:
        return base / model_dir_name
    code = "bnz" if mine_key == "BONANZA" else mine_code.lower()
    model_name = slug(code, "model", model_tag, aug_tag, version, run_date)
    return base / model_name


def split_images_dir(mine: str, mine_code: str | None, split_name: str, split: str = "test") -> Path:
    base = dataset_base_dir(mine, mine_code)
    split_root_name = "splits" if mine.strip().upper() == "BONANZA" else "split"
    return base / split_root_name / split_name / split / "images"


EXPORT_CONTEXT = {
    "mine": "TENGEL",
    "mine_code": "M17",
    "model_tag": "caja",
    "version": "v1",
    "aug_tag": "no_aug",
    "model_group": None,  # Solo BONANZA: no_aug, aug, only.
    "model_dir_name": "m17_model_v1_2026-06-06",  # Nombre exacto opcional; None usa la convención generada.
    "run_date": "2026-06-06",
    "weights_name": "best.pt",
    "yaml": PROJECT_ROOT / "configs" / "caja.yaml",
    "split_name": "m17_split_white_veins_5cm_rocks_v1",
}

RUN_DIR = model_output_dir(
    mine=EXPORT_CONTEXT["mine"],
    mine_code=EXPORT_CONTEXT["mine_code"],
    model_tag=EXPORT_CONTEXT["model_tag"],
    version=EXPORT_CONTEXT["version"],
    run_date=EXPORT_CONTEXT["run_date"],
    aug_tag=EXPORT_CONTEXT["aug_tag"],
    model_group=EXPORT_CONTEXT["model_group"],
    model_dir_name=EXPORT_CONTEXT["model_dir_name"],
)
PT_MODEL_PATH = RUN_DIR / "weights" / EXPORT_CONTEXT["weights_name"]
DATA_YAML_PATH = EXPORT_CONTEXT["yaml"]
TEST_SOURCE = split_images_dir(
    EXPORT_CONTEXT["mine"],
    EXPORT_CONTEXT["mine_code"],
    EXPORT_CONTEXT["split_name"],
    split="test",
)

EXPORT_CFG = {
    "format": "engine",  # TensorRT
    "imgsz": 640,
    "device": 0,
    "dynamic": False,
    "half": True,  # FP16 cuando esté soportado
    "int8": False,  # Usa True solo si preparaste flujo de calibración
    "batch": 1,
    # "workspace": 8,  # Espacio de trabajo opcional de TensorRT en GB
    "data": str(DATA_YAML_PATH),
}

assert PT_MODEL_PATH.exists(), f"Model not found: {PT_MODEL_PATH}"
assert DATA_YAML_PATH.exists(), f"Data yaml not found: {DATA_YAML_PATH}"

print(f"Run dir: {RUN_DIR}")
print(f"PT model: {PT_MODEL_PATH}")
print(f"Data yaml: {DATA_YAML_PATH}")
print(f"Test source: {TEST_SOURCE}")
print(f"Export cfg: {EXPORT_CFG}")


## 4. Exportar `.pt` -> `.engine`


In [ ]:
model = YOLO(str(PT_MODEL_PATH))
export_result = model.export(**EXPORT_CFG)

print("Export output:", export_result)


## 5. Resolver Ruta del `.engine`

Ultralytics normalmente escribe el `.engine` junto al `.pt` con el mismo nombre base.


In [ ]:
ENGINE_PATH = PT_MODEL_PATH.with_suffix(".engine")
print(f"Expected engine path: {ENGINE_PATH}")
print(f"Engine exists: {ENGINE_PATH.exists()}")


## 6. Inferencia de Prueba (Opcional)

Usa una carpeta de imágenes de test para verificar que el engine exportado funciona.


In [ ]:

print(f"Test source: {TEST_SOURCE}")
print(f"Test source exists: {TEST_SOURCE.exists()}")

# Descomenta para correr inferencia con el `.engine` exportado.
# engine_model = YOLO(str(ENGINE_PATH), task="detect")
# engine_pred = engine_model.predict(
#     source=str(TEST_SOURCE),
#     conf=0.25,
#     device=0,
#     save=True,
#     project=str(RUN_DIR.parent),
#     name=f"{RUN_DIR.name}__engine_check",
#     exist_ok=True,
# )


## 7. Comparación Rápida de Velocidad `.pt` vs `.engine` (Opcional)


In [ ]:
import time

# Descomenta para medir la misma fuente con ambos formatos
# pt_model = YOLO(str(PT_MODEL_PATH))
# engine_model = YOLO(str(ENGINE_PATH), task="detect")
#
# t0 = time.time()
# _ = pt_model.predict(source=str(TEST_SOURCE), conf=0.25, device=0, save=False, verbose=False)
# t1 = time.time()
# _ = engine_model.predict(source=str(TEST_SOURCE), conf=0.25, device=0, save=False, verbose=False)
# t2 = time.time()
#
# print(f"PT inference time: {t1 - t0:.2f}s")
# print(f"Engine inference time: {t2 - t1:.2f}s")



## 8. Notas y Solución de Problemas

- Si falla la exportación, verifica compatibilidad CUDA/TensorRT con tu PyTorch y driver GPU instalados.
- Si `int8=True`, asegúrate de que la calibración/data esté bien configurada.
- Si mueves archivos después de exportar, actualiza `EXPORT_CONTEXT` en vez de hardcodear rutas.
- El `.engine` se escribe junto al `.pt` seleccionado, dentro de la carpeta correspondiente `data/.../models/.../weights`.
